# Ultra-Scale Playbook 训练系统 · 第 10/14 课

> 状态：**未开始**  
> 一次只完成一课；未通过前不要打开下一课答案。

## 统一完成标准

代码 4 分、Q1～Q3 各 2 分，通过线 8/10。必须解释正确性边界、显存/通信公式中的单位与分片维度；未实际运行的内容只能标记为静态审查。

# 第 10 课：专家并行 EP

- 对应官方章节：Expert Parallelism
- 前置：第 3 课（all-to-all 概念）、《Switch Transformers》等 MoE 基础（教材要求读者先了解 MoE）
- 状态：未开始

## 本课目标

完成后你需要能够：

- 说明 MoE 层的基本结构与 EP 的切分方式（专家维，不是 token 维）。
- 解释 EP 为什么必须与 DP 组合，以及与 TP 相比"轻量"在哪。
- 推演 token 路由产生的 all-to-all 通信与负载不均问题。
- 解释 DeepSeek-V3 的节点限制路由（每 token ≤ M 个节点）的设计动机。

## 核心概念

### 1. MoE 与 EP

MoE：每一层不再只有一个 FFN，而是多个并行的专家 FFN + 一个 router，每个 token 被路由到少量专家（如 top-2，DeepSeek-V3 每 token 激活 8 个专家）。

**EP：把不同的专家放在不同的 worker 上**（沿专家维切分）。与 TP 的本质区别：

- TP 要切分矩阵乘法本身（需要 all-reduce/all-gather 合并部分和）；
- EP 不用切分 matmul，只需要把 token 的隐藏状态**路由**到正确的专家——通信是 all-to-all（token 的搬迁），而不是部分和。

### 2. 为什么 EP 必须与 DP 组合

EP 只影响 MoE 层。attention、embedding、非 MoE 部分仍然每卡全量计算。如果只做 EP 不并行输入 batch，非 MoE 部分就在所有卡上重复计算。所以：**DP 并行输入（batch），EP 并行专家**，两者组合（教材图：DP×EP 网格）。

另注：有的实现把 EP 看作 DP 的子集——区别在于 EP 用路由而非相同的模型副本处理输入。

### 3. all-to-all：路由通信

- 前向：每个 rank 把自己批次里的 token 按 router 决定的目标专家打包，发给对应 rank（all-to-all）；专家算完，结果再 all-to-all 送回原 rank。
- 通信量 ≈ 被路由的 token 数 × 隐藏状态大小。路由越分散（每个 token 去更多 rank），通信越多。
- 负载不均：若 router 让某个专家拿到远超平均的 token，该 rank 的专家计算时间拖慢整个步（在线程束同步点上被拉齐）。缓解手段：
  - router 辅助损失（aux loss）鼓励均匀路由（Switch 等早期工作）；
  - **节点限制路由**（DeepSeek-V3）：约束每个 token 最多送往 M 个节点（M=4），尽量让 token 留在节点内，把跨节点通信量压到可控水平；
  - 动态专家并行（细粒度专家）等衍生技术。

### 4. 量级例子

DeepSeek-V3：256 个专家、FFN 用细粒度专家切分；每 token 激活 8 个专家。EP 把专家摊到多卡，容量大（总参数量大）但每 token 计算量小——这是 MoE 的容量-计算解耦。

## 具体演示

设 2 个 rank、4 个专家（每 rank 2 个）、批次 16 个 token：

- 均匀路由下每专家 4 token，每 rank 负载 = 8 token 的专家计算，通信 16 token 来回。
- 若 router 失衡（专家 0 拿到 10 token、专家 3 拿到 2 token），rank 0 负载 = 12 token、rank 1 = 4 token → 失衡比 3:1，步时间由 rank 0 决定。

## 代码填空题

模拟专家到 rank 的映射、负载统计与路由通信量。


In [ ]:
def expert_to_rank(num_experts: int, experts_per_rank: int) -> dict[int, int]:
    """专家均匀分到各 rank：rank r 持有专家 [r*e, (r+1)*e)。"""
    mapping = {}
    for e in range(num_experts):
        mapping[e] = ______      # 填空：专家 e 所属的 rank
    return mapping


def load_per_rank(tokens_per_expert: list[int], experts_per_rank: int) -> list[int]:
    """每 rank 的专家负载 = 其专家收到的 token 总数。"""
    num_experts = len(tokens_per_expert)
    rank_load = [0] * (num_experts // experts_per_rank)
    for e, tokens in enumerate(tokens_per_expert):
        r = expert_to_rank(num_experts, experts_per_rank)[e]
        rank_load[r] += ______   # 填空：累加
    return rank_load


def imbalance(loads: list[int]) -> float:
    """失衡比 = 最大负载 / 平均负载。"""
    if not loads or sum(loads) == 0:
        raise ValueError("loads must be non-empty with positive total")
    return ______                # 填空


def routing_volume(tokens_per_expert: list[int]) -> int:
    """
    一次前向的 all-to-all 通信量（token 数）。
    每个 token 都被路由到其目标专家所在 rank——注意目标专家可能就在本地
    rank（这部分不需要网络传输）。严格计算要区分本地/远程；
    这里先返回"被路由的 token 总数"作为上界，远程占比写在注释里。
    """
    return ______                # 填空


if __name__ == "__main__":
    # 均匀路由：每专家 4 token
    tokens = [4, 4, 4, 4]
    print("均匀:", load_per_rank(tokens, 2), "失衡比", f"{imbalance(load_per_rank(tokens, 2)):.2f}")

    # 失衡路由：专家 0 拿到 10 个
    tokens2 = [10, 4, 4, 2]
    loads = load_per_rank(tokens2, 2)
    print("失衡:", loads, "失衡比", f"{imbalance(loads):.2f}")
    print("路由总量:", routing_volume(tokens2), "token")

    # DeepSeek-V3 量级：256 专家、每卡 8 卡节点内放若干专家
    print("专家映射示例:", expert_to_rank(256, 32))   # 每 rank 32 个专家


## 三个问答题


### Q1

为什么说 EP "比 TP 轻量"？两类并行各自传输的数据分别是什么（部分和 vs token 搬运）？为什么 EP 单独使用会造成非 MoE 部分的重复计算，必须与 DP 组合？


### Q2

EP 的 all-to-all 通信发生在什么时机、传输什么数据？DeepSeek-V3 为什么限制"每个 token 最多送往 M=4 个节点"？这个约束与"均匀路由"的目标有什么区别与联系？


### Q3

负载不均为什么是 EP 的致命问题（对比 DP 中负载天然均匀）？请列出至少两种缓解手段，并说明它们各自的代价。极端失衡（某专家 token 数超过容量）时会发生什么？

## 检查与通过标准

总分 10 分：代码正确 4 分（映射、负载、失衡比、路由量）、三题各 2 分、通过线 8 分。

一票否决项：

- 认为 EP 切分的是 token 维或输入 batch（那是 DP/CP）。
- 认为 EP 不需要与 DP 组合。
- 把 EP 通信说成 all-reduce 权重同步（是 all-to-all 路由）。
- 意识不到负载均衡问题及其与路由设计的耦合。


## 官方主参考

- [Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook)
- [PyTorch distributed documentation](https://pytorch.org/docs/stable/distributed.html)